In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os

In [4]:
import zipfile
import os

zip_filename = '/content/animal_data_split.zip'

# Check if it's a valid ZIP file
if not zipfile.is_zipfile(zip_filename):
    raise ValueError(f"❌ {zip_filename} is not a valid ZIP file.")

# Extract
extract_dir = './extracted_zip'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Files extracted to: {extract_dir}")

✅ Files extracted to: ./extracted_zip


## Data Splitting

In [ ]:
import os
import shutil
import random

# Paths
original_dataset_dir = 'facial_expressions'  # Folder containing all class subfolders (e.g., Car, Bike, Bus, etc.)
base_output_dir = f'{original_dataset_dir.strip("/").split("/")[-1]}_split'  # Output folder like: Car-Bike-Dataset_split

train_ratio = 0.8  # 80% train, 20% val
random.seed(42)  # Reproducibility

# Get all class names (folders)
class_names = [d for d in os.listdir(original_dataset_dir) if os.path.isdir(os.path.join(original_dataset_dir, d))]

# Create output directories
for split in ['train', 'val']:
    for class_name in class_names:
        os.makedirs(os.path.join(base_output_dir, split, class_name), exist_ok=True)

# Function to split and copy files for each class
def split_and_copy(category):
    source_folder = os.path.join(original_dataset_dir, category)
    all_files = os.listdir(source_folder)
    random.shuffle(all_files)

    split_index = int(len(all_files) * train_ratio)
    train_files = all_files[:split_index]
    val_files = all_files[split_index:]

    for file in train_files:
        shutil.copy2(os.path.join(source_folder, file), os.path.join(base_output_dir, 'train', category, file))
    for file in val_files:
        shutil.copy2(os.path.join(source_folder, file), os.path.join(base_output_dir, 'val', category, file))

# Process all classes
for class_name in class_names:
    split_and_copy(class_name)

print(f"✅ Dataset successfully split into 'train' and 'val' for {len(class_names)} classes: {class_names}")

## Create Image Generators

**Data Augmentation**

In [5]:
train_dir = "/content/extracted_zip/animal_data_split/train"
val_dir = "/content/extracted_zip/animal_data_split/val"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=20
).flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    # class_mode='binary'
    class_mode='categorical'
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    # class_mode='binary'
    class_mode='categorical'
)

Found 1549 images belonging to 15 classes.
Found 395 images belonging to 15 classes.


In [6]:
print(train_gen.class_indices)

{'Bear': 0, 'Bird': 1, 'Cat': 2, 'Cow': 3, 'Deer': 4, 'Dog': 5, 'Dolphin': 6, 'Elephant': 7, 'Giraffe': 8, 'Horse': 9, 'Kangaroo': 10, 'Lion': 11, 'Panda': 12, 'Tiger': 13, 'Zebra': 14}


In [7]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model layers
base_model.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
x = Dense(15, activation='softmax')(x)
# x = Dense(1, activation='sigmoid')(x)   # Binary classification output

model = Model(inputs=base_model.input, outputs=x)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [9]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    # loss='binary_crossentropy',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 43s 611ms/step - accuracy: 0.0739 - loss: 2.7911 - val_accuracy: 0.0658 - val_loss: 2.7094
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 409ms/step - accuracy: 0.0688 - loss: 2.7050 - val_accuracy: 0.0911 - val_loss: 2.7072
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 383ms/step - accuracy: 0.0832 - loss: 2.7037 - val_accuracy: 0.0709 - val_loss: 2.7096
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 396ms/step - accuracy: 0.0971 - loss: 2.6986 - val_accuracy: 0.1013 - val_loss: 2.6913
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 395ms/step - accuracy: 0.1170 - loss: 2.6903 - val_accuracy: 0.0759 - val_loss: 2.6869
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 383ms/step - accuracy: 0.1222 - loss: 2.6782 - val_accuracy: 0.1367 - val_loss: 2.6821
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 397ms/step - accuracy: 0.1033 - loss: 2.6792 - val_accuracy: 0.1266 - val_loss: 2.6756
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 384ms/step - accuracy: 0.1216 - loss: 2.6758 - val_accu

In [11]:
model.save("resnet50_animal.h5")

In [12]:
from google.colab import files

files.download('/content/resnet50_animal.h5')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>